# Lab 5 — Exercise: Preprocessing the Adult Census Income Dataset

**ITCS 3162 — Introduction to Data Mining**

**Name:** _your name here_
**Date:** _today's date_

**Goal:** prepare the Adult dataset for modeling. We'll predict whether a person's income exceeds $50K/year based on demographic and employment features.

The dataset has the same problems Titanic did, but at scale:
- Missing values (encoded as `?` in the original; sklearn's loader converts them to NaN)
- A mix of numeric, ordinal, and nominal features
- At least one feature that's a thin disguise over another (`education` and `education-num`)
- Features that vary in importance

When you're done, **Restart & Run All**, download as `.ipynb`, and submit via Canvas.


## Setup — run as-is

The primary loader uses `sklearn.fetch_openml`. If that fails (network, etc.), it falls back to a GitHub mirror.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

def load_adult():
    # Primary: sklearn's OpenML mirror (modern, returns clean NaN values)
    try:
        from sklearn.datasets import fetch_openml
        ds = fetch_openml("adult", version=2, as_frame=True, parser="auto")
        X, y = ds.data, ds.target
        df = X.copy()
        df["income"] = y
        return df
    except Exception as e:
        print(f"fetch_openml failed ({type(e).__name__}); trying GitHub fallback...")
        url = ("https://raw.githubusercontent.com/saravrajavelu/Adult-Income-Analysis/"
               "master/adult.data")
        cols = ["age","workclass","fnlwgt","education","education-num",
                "marital-status","occupation","relationship","race","sex",
                "capital-gain","capital-loss","hours-per-week","native-country","income"]
        df = pd.read_csv(url, names=cols, na_values=" ?", skipinitialspace=True)
        return df

df = load_adult()
print(f"Loaded: {df.shape}")
df.head()

## Exercise 1 — Initial scan (10 pts)

In the cell below:
1. Print `df.shape`, `df.info()`, and `df.describe(include="all")` (the latter shows numeric and categorical summaries together).
2. Print the value counts for the target column `income`.

Then answer the questions in the markdown cell.


In [ ]:
# TODO: shape, info, describe, target value counts


**YOUR ANSWERS:**
1. Roughly how balanced is the target?
2. Which columns look numeric, and which look categorical?
3. Are there any obvious ID-like or near-constant columns?



## Exercise 2 — Missing values (15 pts)

Compute:
1. The count of missing values per column (sorted descending).
2. The percentage missing per column (rounded to 2 decimal places).

Then make a decision for each column with missing values:
- If <5% missing → drop those rows
- If 5–50% missing → impute (use mode for categorical, median for numeric)
- If >50% missing → drop the column

Implement your decisions and store the result in `df_clean`. Print its new shape.


In [ ]:
# TODO: explore missing values


# TODO: handle missing values, store in df_clean


**YOUR ANSWER:** Which columns had missing values? What did you do with each, and why?



## Exercise 3 — Find and handle redundant columns (10 pts)

`education` (a string like "Bachelors") and `education-num` (an integer 1–16) carry the same information — one is an ordinal-encoded version of the other.

1. Verify they're truly redundant: produce a crosstab or `groupby` showing the mapping between them.
2. Drop the one you don't want to use (state which and why in the markdown cell). The numeric version is easier to work with — but the string version makes the result more interpretable. Either is defensible; just be deliberate.

Also: the `fnlwgt` column is a Census Bureau sampling weight, **not** a per-person feature. It tells you how representative this row is of the population. For our classification task it's noise. **Drop it.**


In [ ]:
# TODO: verify education vs. education-num are redundant


# TODO: drop redundant column(s) and fnlwgt


**YOUR ANSWER:** Which of `education` and `education-num` did you keep, and why?



## Exercise 4 — Split into train/test before any further preprocessing (5 pts)

Use `train_test_split` with `test_size=0.2`, `random_state=42`, and `stratify` on the income column. Store as `X_train, X_test, y_train, y_test`.

This split must happen **before** scaling and encoding (those will be fit on training data only via the Pipeline at the end).


In [ ]:
from sklearn.model_selection import train_test_split

# TODO: build X (drop income), build y (income), split


## Exercise 5 — Encode the target (5 pts)

The target is currently a string ('>50K' or '<=50K'). For sklearn, convert to 0/1 where 1 = '>50K'. Do this for both `y_train` and `y_test`.

> Note: the OpenML version sometimes returns the target with trailing dots like `>50K.`. Strip whitespace and trailing `.` first if you see them.


In [ ]:
# TODO: encode y_train and y_test to 0/1


## Exercise 6 — Categorical encoding (15 pts)

Identify which categorical columns are **ordinal** (have a natural order — could be encoded as a number) and which are **nominal** (no order — need one-hot). In `X_train` and `X_test`, you should have:
- Nominal: `workclass`, `marital-status`, `occupation`, `relationship`, `race`, `sex`, `native-country`
- Ordinal-by-convention: none cleanly here. (Education would be, but you handled it above by keeping the numeric version.)
- Numeric (already): `age`, `education-num`, `capital-gain`, `capital-loss`, `hours-per-week`

For the **exercise** part (working manually, *not* in a pipeline yet), one-hot encode the nominal columns using `pd.get_dummies()`. Apply it to both `X_train` and `X_test`, and **align** them so they have the same columns afterward.

Hint: after one-hot encoding train and test separately, you can align them with:
```python
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join="left", axis=1, fill_value=0)
```
This guarantees the test set has every column the training set does, filling in zeros for any missing.


In [ ]:
# TODO: one-hot encode nominal columns in X_train and X_test, then align


# TODO: print resulting shapes


## Exercise 7 — Scale numeric features (10 pts)

Using `StandardScaler`:
1. Fit it on the **training set's** numeric columns only.
2. Transform both training and test numeric columns.
3. Store the result as `X_train_final` and `X_test_final`.

Verify that the numeric columns in `X_train_final` have mean ≈ 0 and std ≈ 1.


In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO: scale numeric columns in train and test


## Exercise 8 — Feature importance (15 pts)

Fit a `RandomForestClassifier(n_estimators=200, random_state=42)` on `X_train_final`. Then plot the top 15 features by importance as a horizontal bar chart.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# TODO: fit a random forest, get feature_importances_, plot top 15


**YOUR ANSWER:** Which 3 features look most important? Does that match your intuition?



## Exercise 9 — Build a single Pipeline (15 pts)

Recreate the full preprocessing + model in **one** sklearn Pipeline using `ColumnTransformer`. Use a fresh load of the data (don't carry over earlier transformations).

Your pipeline should:
1. Use `SimpleImputer(strategy="median")` on numeric columns
2. Use `StandardScaler` on numeric columns
3. Use `SimpleImputer(strategy="most_frequent")` then `OneHotEncoder(handle_unknown="ignore")` on nominal columns
4. End with a `LogisticRegression(max_iter=1000)` classifier

Fit it on the original `X_train` (raw, with NaN values still there) and report the test set accuracy.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Recommended: use these column lists
num_cols     = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
nominal_cols = ["workclass", "marital-status", "occupation", "relationship",
                "race", "sex", "native-country"]

# TODO: build numeric_pipe, nominal_pipe, preprocessor (ColumnTransformer), full_pipeline


# TODO: fit on X_train, score on X_test, print accuracy


## Exercise 10 — Reflection (10 pts)

In 4–6 sentences:

1. Which preprocessing step had the biggest impact on the result you got? (Try removing one — e.g., skip scaling — and re-run to see.)
2. Imagine your model will be deployed and run daily on new census records. Why is the **Pipeline approach** safer than the step-by-step approach you did in Exercises 6–8?

YOUR ANSWER:



## Submission checklist

- [ ] Name and date filled in
- [ ] All TODO cells completed and run
- [ ] All `YOUR ANSWER` prompts replaced
- [ ] **Restart & Run All** completes without errors
- [ ] Downloaded as `.ipynb` and uploaded to Canvas
